In [0]:
%sql select * from small_erp_dev.db.products

In [0]:
%sql select * from small_erp_dev.db.customers

In [0]:
%sql select * from small_erp_dev.db.orders

In [0]:
%sql select * from small_erp_dev.db.order_line_items

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW vw_order_line_items
AS
SELECT *
FROM small_erp_dev.db.orders
LEFT JOIN small_erp_dev.db.order_line_items USING (order_id)
LEFT JOIN small_erp_dev.db.products USING (product_id)
LEFT JOIN small_erp_dev.db.customers USING (customer_id)

In [0]:
%sql
SELECT * FROM vw_order_line_items

Databricks visualization. Run in Databricks to view.

In [0]:
%sql
SELECT
    product_name
FROM vw_order_line_items
WHERE order_date > current_date() - INTERVAL 1 MONTH
GROUP BY ALL
ORDER BY SUM(quantity) DESC
LIMIT 1

In [0]:
# Generate a CSV batch of order events for the streaming file source
# (small_erp_dev.db.orders_source). Each row is the full current state of an
# order: a NEW order_id means the order was created, a REPEATED order_id in a
# later row or later file means the order was updated. Downstream can merge on
# order_id keeping the max(updated_at).

VOLUME_PATH = "/Volumes/small_erp_dev/db/orders_source"

batch = [
    # --- initial state of the 7 existing orders (as in small_erp_dev.db.orders)
    "o1,c1,2026-10-01,true,2026-11-19T08:00:00Z",
    "o2,c1,2026-10-15,false,2026-11-19T08:00:01Z",
    "o3,c1,2026-11-01,true,2026-11-19T08:00:02Z",
    "o4,c2,2026-11-05,true,2026-11-19T08:00:03Z",
    "o5,c3,2026-11-06,true,2026-11-19T08:00:04Z",
    "o6,c3,2026-11-07,true,2026-11-19T08:00:05Z",
    "o7,c4,2026-11-10,false,2026-11-19T08:00:06Z",
    # --- updates to existing orders (order_complete flipped to true)
    "o2,c1,2026-10-15,true,2026-11-19T09:05:00Z",
    "o7,c4,2026-11-10,true,2026-11-19T09:10:00Z",
    # --- new orders
    "o8,c1,2026-11-19,false,2026-11-19T09:20:00Z",
    "o9,c2,2026-11-18,false,2026-11-19T09:25:00Z",
    "o10,c4,2026-11-19,true,2026-11-19T09:30:00Z",
]

csv_content = "\n".join(
    ["order_id,customer_id,order_date,order_complete,updated_at"] + batch
) + "\n"

print(csv_content)
dbutils.fs.put(f"{VOLUME_PATH}/orders_batch_001.csv", csv_content, overwrite=True)
print(dbutils.fs.ls(VOLUME_PATH))